In [ ]:
import requests

# Basis-URL des eLabFTW-Servers 
BASE_URL = "YOUR_ELABFTW_BASE_URL"

# API-Key 
API_KEY = "YOUR_API_KEY_HERE"

# Header für die Authentifizierung
headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

In [3]:
headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

user_ids = [214, 215]
limit = 500  # reicht bei 500 Experimenten völlig aus

params = {"limit": limit}
response = requests.get(f"{BASE_URL}/experiments", headers=headers, params=params)

if response.status_code != 200:
    print("Fehler:", response.status_code, response.text)
    exit()

data = response.json()

matching_experiment_ids = []

for exp in data:
    # Prüfen, ob User 214 oder 215 beteiligt ist
    if exp.get("userid") in user_ids:
        relevant = True
    else:
        users = exp.get("users", [])
        relevant = any(u.get("id") in user_ids for u in users)

    if not relevant:
        continue

    # Tags prüfen (z. B. enthält "LLM")
    tags_str = exp.get("tags", "")
    tags = [t.strip().lower() for t in tags_str.split("|") if t.strip()]
    if any("llm" in t for t in tags):
        matching_experiment_ids.append(exp["id"])

print("Experimente mit Tag 'LLM' von User 214 oder 215:")
print(matching_experiment_ids)
print(f"➡️ Anzahl: {len(matching_experiment_ids)}")

Experimente mit Tag 'LLM' von User 214 oder 215:
[2380, 2371, 2370, 2372, 2379, 2378, 2377, 2376, 2375, 2374, 2373, 2369, 2367, 1765, 1762, 1767, 1766, 1769, 1768, 1770, 1772, 1771, 1773, 1774, 1777, 1776, 1779, 1778, 1780, 1781, 1783, 1782, 1786, 1785, 1787, 1788, 1784, 1790, 1789, 1791, 1793, 1792, 1795, 1794, 1840, 1849, 1847, 1856, 1851, 1853, 1858, 1857, 1860, 1859, 1861, 1862, 1864, 1863, 1865, 1875, 1866, 1876, 1878, 1877, 1879, 1881, 1880, 1882, 1872, 1763, 1874, 1873, 1871, 1888, 1887, 1889, 1891, 1890, 1893, 1892, 1894, 1896, 1895, 1898, 1897, 1899, 1901, 1900, 1902, 1883, 1884, 2008, 1886, 2013, 2026, 2014, 2023, 2021, 2020, 1994, 1885, 2019, 2018, 2012, 2007, 2006, 2005, 2001, 1999, 2031, 2034, 2033, 2035, 2036, 2040, 2043, 2041, 2049, 2047, 2050, 2057, 2051, 2058, 2060, 2059, 2062, 2061, 2063, 2065, 2064, 2066, 2068, 2067, 2073, 2074, 2076, 2075, 2078, 2077, 2080, 2079, 2081, 2083, 2082, 2089, 2084, 2090, 2092, 2091, 2093, 2095, 2094, 2097, 2096, 2098, 2099, 2105, 2100, 21

In [4]:
import requests
import json
import re
from bs4 import BeautifulSoup 



headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

exp_id = 1684
response = requests.get(f"{BASE_URL}/experiments/{exp_id}", headers=headers)

if response.status_code != 200:
    print("Fehler:", response.status_code, response.text)
    exit()

exp = response.json()

# --- 1️⃣ Modellname ---
model_name = exp.get("metadata_decoded", {}) \
                .get("extra_fields", {}) \
                .get("LLM Model (whole title)", {}) \
                .get("value", "unknown")

# --- 2️⃣ Audio-ID ---
audio_info = exp.get("items_links", [])
audio_id = audio_info[0]["title"] if audio_info else "unknown"

# --- 3️⃣ Run-ID ---
title = exp.get("title", "")
match = re.search(r"P\d+_\d+_\d+", title)  
run_id = match.group(0) if match else "unknown"

# --- 4️⃣ Tags ---
tags = [t.strip() for t in exp.get("tags", "").split("|") if t.strip()]

# --- 5️⃣ Text extrahieren (nur LLM OUTPUT-Bereich) ---
html = exp.get("body_html", exp.get("body", ""))
soup = BeautifulSoup(html, "html.parser")

# Finde Abschnitt nach <h2>LLM OUTPUT</h2>
llm_section = soup.find("h2", string=re.compile("LLM OUTPUT", re.I))
if llm_section:
    llm_output_html = llm_section.find_next("details")
    text = llm_output_html.get_text(separator="\n").strip() if llm_output_html else ""
else:
    text = soup.get_text(separator="\n").strip()

# --- 6️⃣ Typ bestimmen ---
entry_type = "reference" if "GPT-5" in model_name else "output"

# --- 7️⃣ Alles zusammenführen ---
experiment_data = {
    "experiment_id": exp_id,
    "audio_id": audio_id,
    "model_name": model_name,
    "run_id": run_id,
    "type": entry_type,
    "tags": tags,
    "text": text
}

# --- 8️⃣ Anzeigen oder Speichern ---
print(json.dumps(experiment_data, indent=4, ensure_ascii=False))


{
    "experiment_id": 1684,
    "audio_id": "Anam_Bauchschmerzen",
    "model_name": "Meta Llama 3.1 8B Instruct",
    "run_id": "P1_01_01",
    "type": "output",
    "tags": [
        "AI",
        "LLM",
        "STT",
        "Summary",
        "Meta Llama 3.1 8B Instruct"
    ],
    "text": "LLM Output Starts here:\n\n\n### Patient:\nFrau Milberg, Anke, ist 45 Jahre alt und wurde am 7.12.1974 geboren. Sie ist Bürokauffrau und wohnt zusammen mit ihrem Ehemann und ihren zwei Kindern. Sie hat eine chronische Darmerkrankung (Colitis Ulcerosa) seit 10 Jahren und leidet auch an Zuckerkrankheit und Unterfunktion der Schilddrüse. Sie nimmt L-Tyroxin 75 und Insulin ein und hat eine Insulinpumpe. Sie hat auch eine Nüsseallergie und ist gegen Mitronidazol allergisch. Ihre Eltern haben auch chronische Krankheiten, ihr Vater hat erhöhten Blutdruck und eine Herzkrankheit, und ihre Mutter hat Zuckerkrankheit und eine chronische Darmerkrankheit. Sie hat zwei Schwestern, eine von denen hat auch ei

## Speichern der Zusammenfassungen als json

In [5]:
with open(f"experiment_{exp_id}.json", "w", encoding="utf-8") as f:
    json.dump(experiment_data, f, indent=4, ensure_ascii=False)

In [8]:
headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

def parse_experiment(exp):
    """Extrahiert alle relevanten Felder aus einem einzelnen Experiment."""
    model_name = exp.get("metadata_decoded", {}) \
                    .get("extra_fields", {}) \
                    .get("LLM Model (whole title)", {}) \
                    .get("value", "unknown")

    audio_info = exp.get("items_links", [])
    audio_id = audio_info[0]["title"] if audio_info else "unknown"

    title = exp.get("title", "")
    match = re.search(r"P\d+_\d+_\d+", title)
    run_id = match.group(0) if match else "unknown"

    tags = [t.strip() for t in exp.get("tags", "").split("|") if t.strip()]

    html = exp.get("body_html", exp.get("body", ""))
    soup = BeautifulSoup(html, "html.parser")
    llm_section = soup.find("h2", string=re.compile("LLM OUTPUT", re.I))
    if llm_section:
        llm_output_html = llm_section.find_next("details")
        text = llm_output_html.get_text(separator="\n").strip() if llm_output_html else ""
    else:
        text = soup.get_text(separator="\n").strip()

    entry_type = "reference" if "GPT-5" in model_name else "output"

    return {
        "experiment_id": exp["id"],
        "audio_id": audio_id,
        "model_name": model_name,
        "run_id": run_id,
        "type": entry_type,
        "tags": tags,
        "text": text
    }

# 🔁 Alle Experimente abrufen und verarbeiten
all_data = []

for i, exp_id in enumerate(matching_experiment_ids, start=1):
    response = requests.get(f"{BASE_URL}/experiments/{exp_id}", headers=headers)
    if response.status_code != 200:
        print(f"Fehler bei ID {exp_id}: {response.status_code}")
        continue

    exp = response.json()
    parsed = parse_experiment(exp)
    all_data.append(parsed)

    #print(f"[{i}/{len(matching_experiment_ids)}] verarbeitet: {exp_id} ({parsed['model_name']})")

    #sleep(0.5)  # kleine Pause, um Server nicht zu überlasten

# 💾 Ergebnis speichern 
with open("last_llm_experiments.json", "w", encoding="utf-8") as f:
    for entry in all_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\n✅ Fertig! {len(all_data)} Experimente gespeichert in last_llm_experiments.json")


✅ Fertig! 442 Experimente gespeichert in last_llm_experiments.json


In [ ]:
data = response.json()


In [9]:
#data

## Um die Transkripte diesmal zu speichern, also nicht die Zusammenfassungen

In [ ]:
import requests
import json
import re
from time import sleep
from bs4 import BeautifulSoup

headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

# -------------------------------------------------------------
#  Pagination: ALLE Experimente laden (offset + limit)
# -------------------------------------------------------------
def get_all_experiments():
    all_experiments = []
    offset = 0
    limit = 100
    last_batch = None

    while True:
        #print(f"⬇️ Lade Experimente: offset={offset}, limit={limit}")
        
        resp = requests.get(
            f"{BASE_URL}/experiments",
            headers=headers,
            params={"offset": offset, "limit": limit}
        )

        if resp.status_code != 200:
            print("❌ Fehler beim Laden:", resp.status_code)
            break

        data = resp.json()

        # Keine Daten → fertig
        if not data:
            print("✔ Keine weiteren Daten.")
            break

        # Sicherung: Wenn API immer das Gleiche zurückgibt → abbrechen
        if data == last_batch:
            print("⛔ API gibt immer dieselben Daten zurück — Stop.")
            break

        last_batch = data
        all_experiments.extend(data)

        offset += limit  # nächste Seite

    return all_experiments


# -------------------------------------------------------------
#  Experiment parsing
# -------------------------------------------------------------
def parse_experiment(exp):
    model_name = exp.get("metadata_decoded", {}) \
                    .get("extra_fields", {}) \
                    .get("LLM Model (whole title)", {}) \
                    .get("value", "unknown")

    audio_info = exp.get("items_links", [])
    audio_id = audio_info[0]["title"] if audio_info else "unknown"

    title = exp.get("title", "")
    match = re.search(r"P\d+_\d+_\d+", title)
    run_id = match.group(0) if match else "unknown"

    tags = [t.strip() for t in exp.get("tags", "").split("|") if t.strip()]

    html = exp.get("body_html", exp.get("body", ""))
    soup = BeautifulSoup(html, "html.parser")

    llm_section = soup.find("h2", string=re.compile("LLM OUTPUT", re.I))
    if llm_section:
        llm_output_html = llm_section.find_next("details")
        text = llm_output_html.get_text(separator="\n").strip() if llm_output_html else ""
    else:
        text = soup.get_text(separator="\n").strip()

    entry_type = "reference" if "GPT-5" in model_name else "output"

    return {
        "experiment_id": exp["id"],
        "audio_id": audio_id,
       
       
        "type": entry_type,
        "tags": tags,
        "text": text
    }


# -------------------------------------------------------------
#  1️⃣ ALLE Transkripte LADEN
# -------------------------------------------------------------

all_experiments = get_all_experiments()

# -------------------------------------------------------------
#  2️⃣ NUR: Titel enthält „Transcript Generation“
# -------------------------------------------------------------
pattern = re.compile(r"\btranscript\s+generation\b", re.I)

matching_experiment_ids = [
    exp["id"]
    for exp in all_experiments
    if pattern.search(exp.get("title", ""))
]



# -------------------------------------------------------------
#  3️⃣ Einzelabruf + Parsing
# -------------------------------------------------------------
all_data = []

for i, exp_id in enumerate(matching_experiment_ids, start=1):


    resp = requests.get(f"{BASE_URL}/experiments/{exp_id}", headers=headers)

    if resp.status_code != 200:
        print(f"❌ Fehler bei {exp_id}: HTTP {resp.status_code}")
        continue

    exp = resp.json()
    parsed = parse_experiment(exp)
    all_data.append(parsed)

    #sleep(0.1)  


# -------------------------------------------------------------
#  4️⃣ JSONL speichern
# -------------------------------------------------------------
with open("last_llm_transcripts.json", "w", encoding="utf-8") as f:
    for item in all_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")



✔ Keine weiteren Daten.
